In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.7 MB/s eta 0:00:00


In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
import pydicom
import pandas as pd
import numpy as np
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import zipfile
import shutil

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Fall 2025/ML/Final")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
BEST_MODEL_PATH = CHECKPOINT_DIR / "fasterrcnn_resnet50_fpn_best.pth"

TEST_DATA_PATH = DRIVE_ROOT / "rsna-pneumonia-detection-challenge/stage_2_test_images"

SAMPLE_SUB_PATH = DRIVE_ROOT / "rsna-pneumonia-detection-challenge/stage_2_sample_submission.csv"

LOCAL_TEST_DIR = Path("/content/test_images")

SUBMISSION_PATH = DRIVE_ROOT / "submission.csv"

In [ ]:
NUM_CLASSES = 2
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
CONFIDENCE_THRESHOLD = 0.3

In [ ]:
def setup_test_data():
    has_files = False
    if LOCAL_TEST_DIR.exists():
        if any(LOCAL_TEST_DIR.rglob("*.dcm")):
            has_files = True

    if not has_files:
        if LOCAL_TEST_DIR.exists():
            print(f"Directory {LOCAL_TEST_DIR} exists but contains no DICOMs.")
            shutil.rmtree(LOCAL_TEST_DIR)

        print(f"Setting up Test Data in {LOCAL_TEST_DIR}")
        LOCAL_TEST_DIR.mkdir(parents=True, exist_ok=True)

        if TEST_DATA_PATH.exists() and TEST_DATA_PATH.is_dir():
            print(f"Found folder at {TEST_DATA_PATH}")
            print("Copying images to local runtime")

            cmd = f"cp -r -n '{str(TEST_DATA_PATH)}/.' '{str(LOCAL_TEST_DIR)}'"
            exit_code = os.system(cmd)

            if exit_code == 0:
                print("Copy Complete.")
            else:
                print("System copy failed")
                for item in TEST_DATA_PATH.iterdir():
                    if item.is_file():
                        shutil.copy2(item, LOCAL_TEST_DIR)
                print("Copy Complete.")

        elif (DRIVE_ROOT / "rsna-pneumonia-detection-challenge/stage_2_test_images.zip").exists():
            zip_path = DRIVE_ROOT / "rsna-pneumonia-detection-challenge/stage_2_test_images.zip"
            print(f"Folder not found, but found zip at {zip_path}. Extracting...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(LOCAL_TEST_DIR)
            print("Extraction Complete.")

        else:
            raise FileNotFoundError(f"Cannot find test data at {TEST_DATA_PATH} (folder) or as a zip.")
    else:
        print("Test data already setup in local runtime.")

In [ ]:
class RSNATestDataset(Dataset):
    def __init__(self, img_dir, img_size=512):
        self.img_dir = Path(img_dir)
        self.img_size = img_size
        self.file_list = list(self.img_dir.rglob("*.dcm"))

        if len(self.file_list) == 0:
            raise RuntimeError(f"No .dcm files found in {self.img_dir}")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        dcm_path = self.file_list[idx]
        patient_id = dcm_path.stem

        dcm = pydicom.dcmread(dcm_path)
        img_arr = dcm.pixel_array

        orig_h, orig_w = img_arr.shape

        img_pil = Image.fromarray(img_arr).convert("RGB")
        img_resized = img_pil.resize((self.img_size, self.img_size), resample=Image.BILINEAR)

        img_np = np.array(img_resized) / 255.0
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float()

        return img_tensor, patient_id, orig_w, orig_h

In [ ]:
def get_model():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model

In [ ]:
if __name__ == "__main__":
    setup_test_data()

    if not BEST_MODEL_PATH.exists():
        print(f"Warning: {BEST_MODEL_PATH} not found")
        BEST_MODEL_PATH = CHECKPOINT_DIR / "fasterrcnn_resume.pth"

    model = get_model()
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(DEVICE)
    model.eval()

    test_ds = RSNATestDataset(LOCAL_TEST_DIR, img_size=512)
    test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

    predictions_map = {}

    print(f"Running Inference on {len(test_ds)} images found locally")

    with torch.no_grad():
        for images, patient_ids, orig_ws, orig_hs in tqdm(test_dl):
            images = list(img.to(DEVICE) for img in images)
            outputs = model(images)

            for i, output in enumerate(outputs):
                patient_id = patient_ids[i]
                orig_w = orig_ws[i].item()
                orig_h = orig_hs[i].item()

                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()

                keep = scores > CONFIDENCE_THRESHOLD
                final_boxes = boxes[keep]
                final_scores = scores[keep]

                prediction_string = ""

                if len(final_boxes) > 0:
                    preds = []
                    for box, score in zip(final_boxes, final_scores):
                        scale_x = orig_w / 512.0
                        scale_y = orig_h / 512.0

                        x_min = box[0] * scale_x
                        y_min = box[1] * scale_y
                        x_max = box[2] * scale_x
                        y_max = box[3] * scale_y

                        width = x_max - x_min
                        height = y_max - y_min
                        x = x_min
                        y = y_min

                        preds.append(f"{score:.4f} {x:.1f} {y:.1f} {width:.1f} {height:.1f}")

                    prediction_string = " ".join(preds)

                predictions_map[patient_id] = prediction_string

    if SAMPLE_SUB_PATH.exists():
        sample_df = pd.read_csv(SAMPLE_SUB_PATH)
        print(f"Loaded Sample Submission with {len(sample_df)} rows.")

        final_results = []
        missing_count = 0

        for pid in sample_df['patientId']:
            if pid in predictions_map:
                pred = predictions_map[pid]
            else:
                pred = ""
                missing_count += 1

            final_results.append({"patientId": pid, "PredictionString": pred})

        if missing_count > 0:
            print(f"Warning: {missing_count} patients from sample submission were not found in image folder. They will have empty predictions.")

        df = pd.DataFrame(final_results)
    else:
        print("WARNING: Sample submission file not found. Generating CSV based ONLY on images found.")
        df = pd.DataFrame([{"patientId": k, "PredictionString": v} for k, v in predictions_map.items()])

    df.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSubmission saved to {SUBMISSION_PATH}")